# Text-detection experiments
Run these cells **after** you have placed a labelled dataset at `../data/processed/dataset.csv`
(columns `text,label,source,generator,domain`; label 0 = human, 1 = AI).
This notebook contains no pre-computed results: every number is produced when you run it.
Slow cells (RoBERTa / perplexity) process texts one by one on CPU; cap sizes with `MAX_*`.

In [ ]:
import sys, json
sys.path.insert(0, "../backend")
import numpy as np, pandas as pd
from app.data import load_dataset_any, split_random, split_holdout, check_leakage, describe
from app.evaluation import compute_metrics, metrics_with_abstention, save_results
DATA = "../data/processed/dataset.csv"
OUT = "../results"
SEED = 42
MAX_TRAIN, MAX_VAL, MAX_TEST = 2000, 500, 500   # lower for a quick CPU run
UNSEEN_GENERATOR = None                          # e.g. ["gpt-4"] to evaluate on an unseen generator

## Load data and split (leakage-aware)

In [ ]:
df = load_dataset_any(DATA)
print(json.dumps(describe(df), indent=2))
if UNSEEN_GENERATOR:
    train, val, test = split_holdout(df, "generator", UNSEEN_GENERATOR, seed=SEED); split = "unseen-generator"
else:
    train, val, test = split_random(df, seed=SEED); split = "random"
cap = lambda d, n: d.sample(n=n, random_state=SEED).reset_index(drop=True) if len(d) > n else d
train, val, test = cap(train, MAX_TRAIN), cap(val, MAX_VAL), cap(test, MAX_TEST)
print("leakage:", check_leakage(train, val, test), "| split:", split, "| sizes:", len(train), len(val), len(test))
ytr, yva, yte = train.label.values, val.label.values, test.label.values

## Experiment A - TF-IDF + Logistic Regression

In [ ]:
from app.feature_model import FeatureTextClassifier
A = FeatureTextClassifier.train(train.text, ytr, seed=SEED)
pA = A.predict_proba(test.text)
mA = compute_metrics(yte, pA); save_results("A_tfidf_logreg", mA, OUT, yte, pA, {"split": split})
mA

## Experiment B - pretrained RoBERTa baseline (not our contribution)

In [ ]:
from app.config import Settings
from app.pipeline import TextDetectionPipeline
pipe = TextDetectionPipeline.from_settings(Settings.from_env())
sig_test = [pipe.compute_signals(t)[0] for t in test.text]
pB = np.array([s.roberta_ai if s.roberta_ai is not None else np.nan for s in sig_test]); ok = ~np.isnan(pB)
mB = compute_metrics(yte[ok], pB[ok]); save_results("B_roberta_baseline", mB, OUT, yte[ok], pB[ok], {"split": split})
mB

## Experiment C - linguistic features + log-perplexity

In [ ]:
from app.feature_model import NumericFeatureClassifier, linguistic_matrix
from app.features import FEATURE_NAMES
def feat(texts):
    L = linguistic_matrix(texts)
    ppl = np.array([(pipe.perplexity.compute(t).perplexity or np.nan) for t in texts], float)
    return np.column_stack([L, np.log(ppl)])
cols = list(FEATURE_NAMES) + ["log_perplexity"]
Xtr, Xte = feat(list(train.text)), feat(list(test.text))
fill = np.nanmedian(Xtr, axis=0)
Xtr, Xte = np.where(np.isnan(Xtr), fill, Xtr), np.where(np.isnan(Xte), fill, Xte)
C = NumericFeatureClassifier.train(Xtr, ytr, cols, seed=SEED)
pC = C.predict_proba(Xte)
mC = compute_metrics(yte, pC); save_results("C_linguistic_perplexity_logreg", mC, OUT, yte, pC, {"split": split, "coefficients": C.coefficients()})
mC

## Experiment D - fusion (prototype weights vs weights learned on the validation split)

In [ ]:
from app.scoring import COMPONENTS, FusionConfig, component_scores, fit_fusion_weights, fuse
cfg0 = pipe.fusion_cfg
sig_val = [pipe.compute_signals(t)[0] for t in val.text]
M = lambda sigs: np.array([[component_scores(s, cfg0)[c] if component_scores(s, cfg0)[c] is not None else np.nan for c in COMPONENTS] for s in sigs], float)
p_proto = np.array([fuse(s, cfg0).detection_score for s in sig_test])
fitted = fit_fusion_weights(M(sig_val), yva, cfg0)
p_fit = np.array([fuse(s, fitted).detection_score for s in sig_test])
print("prototype weights:", cfg0.weights()); print("learned weights:  ", fitted.weights())
save_results("D_fusion_prototype_weights", compute_metrics(yte, p_proto), OUT, yte, p_proto, {"split": split})
save_results("D_fusion_learned_weights", compute_metrics(yte, p_fit), OUT, yte, p_fit, {"split": split})
pd.read_csv(f"{OUT}/results_summary.csv")

## Calibration (feeds the Trust Score)
Fit on the **validation** split, then check on the **test** split. Save the artifacts to `../data/processed/` to activate them.

In [ ]:
from app.trust_score import build_calibration_report, expected_calibration_error, brier_score
val_scores = [fuse(s, fitted).detection_score for s in sig_val]
report, calibrator = build_calibration_report(val_scores, yva, dataset=DATA + " (validation)")
test_scores = [fuse(s, fitted).detection_score for s in sig_test]
print("test ECE before:", expected_calibration_error(test_scores, yte), "after:", expected_calibration_error([calibrator(p) for p in test_scores], yte))
print("test Brier before:", brier_score(test_scores, yte), "after:", brier_score([calibrator(p) for p in test_scores], yte))
# To activate:  report.save('../data/processed/calibration.json')   fitted.save('../data/processed/fusion_config.json')

## Robustness
Provide real modified texts (paraphrased AI, human-edited AI, unseen generator, other domain) in a CSV with columns
`original_text, modified_text, perturbation_type`.

In [ ]:
from app.robustness import compare_texts, summarize
pairs = pd.read_csv("../data/test/pairs.csv")   # you supply this file
res = [compare_texts(pipe.analyze, r.original_text, r.modified_text, getattr(r, "perturbation_type", None)) for r in pairs.itertuples()]
pd.DataFrame([r.to_dict() for r in res]).groupby("perturbation_type").agg(
    pairs=("prediction_changed", "size"), changed=("prediction_changed", "mean"), mean_prob_change=("probability_change", "mean"))

## Experiment E (later) - fine-tuned DeBERTa
Train from the command line (GPU recommended), then evaluate:

    python -m training.train --data ../data/processed/dataset.csv --output-dir ../data/processed/models/deberta
    python -m scripts.run_experiments --data ../data/processed/dataset.csv --experiments E --checkpoint ../data/processed/models/deberta